## Shell脚本

```shell
#!/bin/bash

# 默认选项
start=false
model=""
openai=false

# 解析命令行参数
for arg in "$@"; do
  case $arg in
    --start)
      start=true
      ;;
    --model=*)
      model=${arg#--model=}
      ;;
    --stopworker)
      stopworker=true
      ;;
    --stop)
      stop=true
      ;;
    --reload)
      reload=true
      ;;
    --help)
      help=true
      ;;
    --openai)
      openai=true
      ;;
  esac
done


# 检查是否传入了正确的参数
if [ "$start" = true ]; then
  # 启动 FastChat
  echo "启动 FastChat..."
  source /root/miniconda3/bin/activate textgen
  nohup python3 -m fastchat.serve.controller >/dev/null 2>&1 &
  echo
fi

if [ -n "$model" ]; then
  # 启动特定模型
  echo "启动模型: $model"
  source /root/miniconda3/bin/activate textgen

  if [ "$openai" = true ];then
     echo "$openai"
     nohup python3 -m fastchat.serve.model_worker --model-names "gpt-3.5-turbo,text-davinci-003,text-embedding-ada-002" --model-path "/data/models/$model" >/dev/null 2>&1 &
  else
     nohup python3 -m fastchat.serve.model_worker --model-path "/data/models/$model" >/dev/null 2>&1 &
  fi
  echo "等待模型加载..."
  sleep 30
  nohup python3 -m fastchat.serve.gradio_web_server >/dev/null 2>&1 &
  nohup python3 -m fastchat.serve.openai_api_server --host 0.0.0.0 --port 8000  >/dev/null 2>&1 &

fi

if [ "$reload" = true ]; then
  echo "重启 gradio API ..."
  ps -ef |grep fastchat|grep gradio_web_server |awk '{print $2}'|xargs -i kill -9 {}
  ps -ef |grep fastchat|grep openai_api_server |awk '{print $2}'|xargs -i kill -9 {}
  nohup python3 -m fastchat.serve.gradio_web_server >/dev/null 2>&1 &
  nohup python3 -m fastchat.serve.openai_api_server --host 0.0.0.0 --port 8000  >/dev/null 2>&1 &

fi

if [ "$stopworker" = true ]; then
  # 关闭worker
  echo "关闭所有worker..."
  ps -ef |grep fastchat|grep model_worker |awk '{print $2}'|xargs -i kill -9 {}
  ps -ef |grep fastchat|grep gradio_web_server |awk '{print $2}'|xargs -i kill -9 {}
  ps -ef |grep fastchat|grep openai_api_server |awk '{print $2}'|xargs -i kill -9 {}
fi

if [ "$stop" = true ]; then
  # 关闭模型
  echo "关闭所有服务..."
  ps -ef |grep fastchat|awk '{print $2}'|xargs -i kill -9 {}
  echo "服务关闭"
fi

if [ "$help" = true ]; then
  echo """
  --start:               启动 controller / gradio_web_server。
  --model=<model_name>:  指定需要启动的model 名称。
    --openai:            符合openai 的名称规范以满足openAI 的sdk调用
  --stopworker:          关闭所有worker
  --reload:              重启 gradio API
  --stop:                关闭所有服务
  """
fi
```

## 服务启动


- start controller
```
fastchat --start
```

- 对接 openai SDK (gradio 无法使用)
```
fastchat --model=phoenix-inst-chat-7b --openai
```

- 启动 api + gradio
```
fastchat --model=phoenix-inst-chat-7b
```



## LangChain + FastChat

```python
import os

os.environ["OPENAI_API_BASE"] = "http://localhost:8000/v1"
os.environ["OPENAI_API_KEY"] = "EMPTY"


from langchain.embeddings import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()

text = "This is a test document."

query_result = embeddings.embed_query(text)

doc_result = embeddings.embed_documents([text])

query_result
```